In [1]:
import os

try:
    import google.colab
    IS_COLAB = True
    print("Google Colab")
except ImportError:
    IS_COLAB = False
    print("로컬 Jupyter")

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPEN_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("import 완료")
print("API 키 OK" if os.environ.get("OPENAI_API_KEY") else "키 없음")

로컬 Jupyter
import 완료
API 키 OK


In [4]:
from langchain_openai import ChatOpenAI 
from langchain_core.prompts import ChatPromptTemplate

"""LLM 설정 코드"""
llm = ChatOpenAI(
    model="gpt-4o-mini",   
    temperature=0.3    
)

PERSONAS = {
    "분석가": "당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.",
    "마케터": "당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.",
    "개발자": "당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.",
    "기획자": "당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.",
    "디자이너": "당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.",
}

prompt = ChatPromptTemplate.from_messages([
    ("system", "{persona}"),
    ("human", "{question}")
])

chain = prompt | llm

def ask(persona_name: str, question: str) -> str:
    system_message = PERSONAS[persona_name]

    result = chain.invoke({
        "persona": system_message,
        "question": question
    })

    return result.content

def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)    # "/change 마케터" → ["/change", "마케터"]
    cmd = parts[0]

    if cmd in ("/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True       # 종료 플래그 True

    if cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")

    elif cmd == "/help":
        print("📖 명령어: /quit, /exit, /change <이름>, /persona, /help")

    elif cmd == "/list":
        for name, description in PERSONAS.items():
            print(f"- {name}: {description}")

    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False         # 페르소나만 변경
        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")

    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False

def run_chatbot(initial_persona: str = "분석가"):
    persona = initial_persona
    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")

    while True:
        user_input = input("질문> ")
        user_input = user_input.strip()
        
        if not user_input:                       # 빈 입력은 무시
            continue

        if user_input.startswith("/"):           # 명령어 처리
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                break
        else:                                    # 일반 질문 처리
            answer = ask(persona, user_input)
            print(f"[{persona}] {answer}\n")


In [5]:
run_chatbot()

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

- 분석가: 당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.
- 마케터: 당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.
- 개발자: 당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.
- 기획자: 당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.
- 디자이너: 당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.
📖 명령어: /quit, /exit, /change <이름>, /persona, /help
👋 종료합니다.
